# Building the opportunities table

In this notebook I turn the raw data into one table with one row per *opportunity*: a single to the outfield with a runner on second, where the third-base coach has to decide whether to send the runner home or hold him at third.

For each opportunity I work out whether the runner was sent and whether he was safe, and I add the facts the model uses: the ball's direction, the runner's speed and the outfielder's arm strength. The table is saved as `opportunities.csv`, which `model.ipynb` reads.

I use three files from the `pulls/` scripts:

- `statcast_game_data.csv`: every regular-season plate appearance from 2022–2026
- `sprint_speed.csv`: each runner's sprint speed by season, from Baseball Savant
- `arm_strength.csv`: each fielder's arm strength by season, downloaded from Baseball Savant

## Setup

Savant measures sprint speed from a runner's fastest runs. I only trust a runner's number if he had at least `MIN_RUNS` of those runs in the season.

In [ ]:
import numpy as np
import pandas as pd

MIN_RUNS = 10

## Find the opportunities

An opportunity is a plate appearance where:

- the batter hit a single,
- an outfielder fielded it (`hit_location` 7, 8 or 9 is left, center or right field), and
- there was a runner on second (`on_2b` holds his player ID).

The away team bats in the top of the inning and the home team bats in the bottom, which tells me the batting team.

In [ ]:
games = pd.read_csv("../statcast_game_data.csv")
games["season"] = games.game_date.str[:4].astype(int)

plays = games[(games.events == "single") & games.hit_location.isin([7, 8, 9]) & games.on_2b.notna()]
plays["on_2b"] = plays.on_2b.astype(int)
plays["bat_team"] = plays.home_team.where(plays.inning_topbot == "Bot", plays.away_team)

## Was the runner sent, and was he safe?

Public data doesn't say what the coach did, so I work it out from two things: how many runs scored on the play, and whether the play description says someone was "out at home".

| Situation | Scored | Out | Stopped |
|---|---|---|---|
| Third base empty | 1 or more runs | 0 runs and "out at home" | anything else |
| Runner on third | 2 or more runs | 1 run and "out at home" | anything else |

With a runner on third, that runner scores first. So if only one run scored and someone was out at home, it was the runner from second who was thrown out.

- `sent`: the runner scored or was thrown out at home
- `safe`: 1 if a sent runner scored, 0 if he was out. I leave it blank for held runners, because we never find out what would have happened.

"Stopped" is my stand-in for "held". It also includes runners who never had a real chance to score, and I can't tell whether the coach waved the runner or the runner decided on his own.

In [ ]:
runs = plays.post_bat_score - plays.bat_score
runs_needed = plays.on_3b.notna().map({True: 2, False: 1})
out_at_home = plays.des.str.contains("out at home", na=False)

plays["outcome"] = "stopped"
plays.loc[runs >= runs_needed, "outcome"] = "scored"
plays.loc[(runs == runs_needed - 1) & out_at_home, "outcome"] = "out"

plays["sent"] = plays.outcome != "stopped"
plays["safe"] = (plays.outcome == "scored").astype(float).where(plays.sent)

## Which direction was the ball hit?

Statcast records where each ball was picked up as a point (`hc_x`, `hc_y`) on its own field chart, with home plate at about (125.42, 198.27). I turn that point into a *spray angle*: the angle between the ball's path and a straight line from home plate through second base. About −30 is left field, 0 is straight up the middle and +30 is right field.

I checked this against the data: the average spray angle is about −30 for balls fielded by left fielders, 0 for center fielders and +30 for right fielders.

The foul lines are at ±45. A few balls land just past them because of small errors in the coordinates, so I pull those back to ±45. Anything past ±60 is a data error, so I left it blank.

Note: Found the formula on the internet via Gemini and double checked the values myself

In [ ]:
angle = np.degrees(np.arctan2(plays.hc_x - 125.42, 198.27 - plays.hc_y))
plays["spray_angle"] = angle.where(angle.abs() <= 60).clip(-45, 45)

## How fast is the runner?

I match each runner on second to his sprint speed for that season, in feet per second. If he doesn't have a trusted number, usually because he didn't run enough that season, I use the season's median sprint speed instead. `speed_filled_in` marks those plays.

In [ ]:
speed = pd.read_csv("../sprint_speed.csv")
speed = speed[speed.competitive_runs >= MIN_RUNS]
speed = speed.rename(columns={"player_id": "on_2b", "sprint_speed": "runner_speed"})
plays = plays.merge(speed[["on_2b", "season", "runner_speed"]], on=["on_2b", "season"], how="left")

plays["speed_filled_in"] = plays.runner_speed.isna()
median_speed = speed.groupby("season").runner_speed.median()
plays["runner_speed"] = plays.runner_speed.fillna(plays.season.map(median_speed))

## How strong is the outfielder's arm?

`hit_location` tells me which outfielder fielded the ball, and `fielder_7`, `fielder_8` and `fielder_9` hold the player IDs of the left, center and right fielders. I use those to find the outfielder who made the play and match him to his outfield arm strength for that season, in mph. If he doesn't have one, I use the season's median instead. `arm_filled_in` marks those plays.

In [ ]:
plays["fielder"] = plays.fielder_8
plays.loc[plays.hit_location == 7, "fielder"] = plays.fielder_7
plays.loc[plays.hit_location == 9, "fielder"] = plays.fielder_9

arm = pd.read_csv("../arm_strength.csv")
arm = arm.rename(columns={"player_id": "fielder", "arm_of": "fielder_arm"})
plays = plays.merge(arm[["fielder", "season", "fielder_arm"]], on=["fielder", "season"], how="left")

plays["arm_filled_in"] = plays.fielder_arm.isna()
median_arm = arm.groupby("season").fielder_arm.median()
plays["fielder_arm"] = plays.fielder_arm.fillna(plays.season.map(median_arm))

## Save

A few plays are missing exit velocity, launch angle, batted-ball type or spray angle. I mark the plays that have all four as `complete`, and the model only uses those.

I save only the columns the other notebooks use, then show how many runners scored, were thrown out or stopped in each season.

In [ ]:
plays["complete"] = plays[["launch_speed", "launch_angle", "bb_type", "spray_angle"]].notna().all(axis=1)

columns = ["game_pk", "game_date", "season", "inning", "at_bat_number", "bat_team", "outs_when_up",
           "bb_type", "launch_speed", "launch_angle", "spray_angle", "runner_speed", "speed_filled_in",
           "fielder_arm", "arm_filled_in", "complete", "outcome", "sent", "safe", "des"]
plays[columns].to_csv("../opportunities.csv", index=False)

pd.crosstab(plays.season, plays.outcome)